# Strands Agent com Tutorial de AgentCore Memory usando Hooks

## Visão Geral

Este tutorial demonstra como construir um assistente pessoal inteligente usando Strands agents integrados com AgentCore Memory através de hooks. O agente mantém o contexto da conversa e aprende com as interações para fornecer respostas personalizadas.

## Detalhes do Tutorial

**Caso de Uso**: Assistente de Matemática

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional de longo prazo                                                    |
| Tipo de agente      | Assistente de Matemática                                                         |
| Framework agêntico  | Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes         | AgentCore Summary Strategy para Memory, Hooks para armazenar e recuperar Memory  |
| Complexidade        | Intermediário                                                                    |


Você aprenderá a:
- Configurar o AgentCore Memory com resumos de conversas
- Criar hooks de memória para armazenamento e recuperação automáticos
- Construir um Strands agent com memória persistente
- Testar a funcionalidade de memória entre conversas
- Usar ramificação de conversas para caminhos de aprendizado alternativos
- Aplicar metadata para rastrear o progresso e desempenho do estudante

### Contexto do Cenário

Neste exemplo, você criará um Assistente de Matemática onde armazenará resumos das conversas anteriores.
Principais funcionalidades deste exemplo:
- **Armazenamento Automático de Memória**: As conversas são salvas automaticamente
- **Recuperação de Contexto**: Conversas anteriores informam as respostas atuais
- **Geração de Resumo**: Informações-chave são extraídas e resumidas
- **Integração de Ferramentas**: Ferramenta de calculadora para operações matemáticas
- **Ramificação de Conversas**: Explore níveis de dificuldade alternativos e abordagens de ensino
- **Rastreamento de Metadata**: Marque eventos com dificuldade, desempenho e marcos de aprendizado

## Arquitetura
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Pré-requisitos

Para executar este tutorial, você precisará de:
- Python 3.10+
- Credenciais AWS com permissões de Amazon Bedrock AgentCore Memory
- Amazon Bedrock AgentCore SDK

## Passo 1: Configuração do ambiente
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para fazer este notebook funcionar.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.memory.manager import Memory, MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    SemanticStrategy
)
from bedrock_agentcore.memory import MemorySessionManager
from bedrock_agentcore.memory.constants import (
    ConversationalMessage, MessageRole, RetrievalConfig
)
from bedrock_agentcore.memory.models import (
    StringValue, EventMetadataFilter, LeftExpression, RightExpression, OperatorType
)


In [ ]:
import os
import logging
from typing import List, Dict
from strands import Agent
from datetime import datetime
from strands_tools import calculator
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("memory-tutorial")

# Configuration - replace with your values
REGION = os.getenv('AWS_REGION', 'us-west-2')
ACTOR_ID = f"student-{datetime.now().strftime('%Y%m%d%H%M%S')}"
SESSION_ID = f"math-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Define message role constants for cleaner code
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

## Passo 2: Criar Recurso de Memória

Neste passo, estamos criando nosso recurso de memória com uma estratégia semântica. Este recurso irá armazenar e organizar nossos dados de conversa. A SemanticStrategy integrada captura automaticamente fatos das conversas sem exigir uma IAM execution role.


In [ ]:
from botocore.exceptions import ClientError

# Initialize Memory Manager
memory_manager = MemoryManager(region_name=REGION)
memory_name = "MathAssistant"

# Define memory strategy using SemanticStrategy
strategies = [
    SemanticStrategy(
        name="MathLearningMemory",
        description="Captures facts from math learning conversations",
        namespaces=["/students/math/{actorId}/"]
    )
]

# Create memory resource using MemoryManager
memory_id = None  # Initialize to avoid NameError in exception handler
try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=strategies,
        description="Memory for tutorial agent",
        event_expiry_days=30
    )
    memory_id = memory.id
    logger.info(f"✅ Created memory: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.error(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            memory_manager.delete_memory(memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")
    # Re-raise the exception to stop execution
    raise

# Verify memory_id was successfully obtained
if memory_id is None:
    raise RuntimeError("Failed to create or retrieve memory ID")

## Passo 3: Inicializar o Session Manager

Agora vamos criar um MemorySessionManager e uma MemorySession para nosso estudante. O session manager fornece uma API mais limpa ao lidar automaticamente com os parâmetros memory_id, actor_id e session_id em todas as operações.

Essa abordagem baseada em sessão simplifica as operações de memória e torna o código mais sustentável.

In [ ]:
# Initialize the session manager
session_manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)

# Create a memory session for the specific student
student_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID,
    session_id=SESSION_ID
)

logger.info(f"✅ Session manager initialized for memory: {memory_id}")
logger.info(f"✅ Student session created for actor: {ACTOR_ID}")
logger.info(f"   Session ID: {SESSION_ID}")

## Passo 4: Criar o Memory Hook Provider com Suporte a Sessão

Este passo define nossa classe personalizada `MemoryHookProvider` que automatiza as operações de memória usando a MemorySession. Hooks são funções especiais que executam em pontos específicos do ciclo de vida de execução de um agente. O hook de memória que estamos criando serve duas funções principais:

1. **Recuperar Memórias**: Busca automaticamente conversas passadas relevantes quando um usuário envia uma mensagem usando `search_long_term_memories()`
2. **Salvar Memórias**: Armazena novas conversas após o agente responder usando `add_turns()` com objetos ConversationalMessage

Isso cria uma experiência de memória fluida sem gerenciamento manual, e a API baseada em sessão elimina a necessidade de passar memory_id, actor_id e session_id repetidamente.

In [ ]:
class MemoryHookProvider(HookProvider):
    """Hook provider for automatic memory management using MemorySession"""
    
    def __init__(self, student_session):
        """Initialize with a MemorySession instance
        
        Args:
            student_session: MemorySession instance for the student
        """
        self.student_session = student_session
        
        # Define retrieval configuration for math learning context
        self.retrieval_config = RetrievalConfig(
            top_k=5,
            relevance_score=0.3
        )
    
    def retrieve_memories(self, event: MessageAddedEvent):
        """Retrieve relevant memories before processing user message using MemorySession"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_message = messages[-1]["content"][0].get("text", "")
            
            try:
                # Use MemorySession for context retrieval (no need to pass actor_id)
                namespace_prefix = f"/students/math/{self.student_session._actor_id}/"
                
                # Search long-term memories using session API
                memories = self.student_session.search_long_term_memories(
                    query=user_message,
                    namespace_prefix=namespace_prefix,
                    top_k=self.retrieval_config.top_k
                )
                
                # Filter by relevance score
                filtered_memories = [
                    memory for memory in memories
                    if memory.get("score", 0) >= self.retrieval_config.relevance_score
                ]
                
                # Extract memory content
                memory_context = []
                for memory in filtered_memories:
                    if isinstance(memory, dict):
                        content = memory.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            score = memory.get('score', 0)
                            if text:
                                memory_context.append(f"[Score: {score:.2f}] {text}")
                
                # Inject memories into user message
                if memory_context:
                    context_text = "\n".join(memory_context)
                    original_text = messages[-1]["content"][0].get("text", "")
                    messages[-1]["content"][0]["text"] = (
                        f"{original_text}\n\nStudent Learning Context:\n{context_text}"
                    )
                    logger.info(f"✅ Retrieved {len(memory_context)} relevant memories (filtered from {len(memories)} total)")
                    
            except Exception as e:
                logger.error(f"Failed to retrieve memories: {e}")
    
    def save_memories(self, event: AfterInvocationEvent):
        """Save conversation after agent response using MemorySession"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last user and assistant messages
                user_msg = None
                assistant_msg = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg and "toolResult" not in msg["content"][0]:
                        user_msg = msg["content"][0]["text"]
                        break
                
                if user_msg and assistant_msg:
                    # Use MemorySession with ConversationalMessage objects
                    interaction_messages = [
                        ConversationalMessage(user_msg, USER),
                        ConversationalMessage(assistant_msg, ASSISTANT)
                    ]
                    
                    result = self.student_session.add_turns(interaction_messages)
                    logger.info(f"✅ Saved conversation using MemorySession - Event ID: {result['eventId']}")
                    
        except Exception as e:
            logger.error(f"Failed to save memories: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_memories)
        registry.add_callback(AfterInvocationEvent, self.save_memories)
        logger.info("✅ Memory hooks registered with MemorySession support")

## Passo 5: Criar Agente com Memória

Agora estamos criando nosso Strands agent e conectando-o com nosso hook provider de memória que usa MemorySession. Este agente terá duas capacidades principais:

1. **Integração de Memória**: Os hooks de memória que criamos habilitarão a recuperação automática de contexto usando operações baseadas em sessão
2. **Ferramenta Calculadora**: O agente pode realizar operações matemáticas quando necessário

Essa combinação cria um tutor de matemática que tanto lembra do progresso do estudante quanto pode realizar cálculos úteis.

In [ ]:
# Create memory hook provider with MemorySession
memory_hooks = MemoryHookProvider(student_session)

# Create agent with memory hooks and calculator tool
agent = Agent(
    hooks=[memory_hooks],
    model = "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[calculator],
    system_prompt="You are a helpful personal math tutor. You assist users in solving math problems and provide personalized assistance based on their learning progress and preferences.",
)

logger.info("✅ Agent created with MemorySession-based hooks")
logger.info(f"   Student: {ACTOR_ID}")
logger.info(f"   Session: {SESSION_ID}")

**Nosso agente está configurado! Vamos testá-lo agora.**

## Testar Funcionalidade de Memória

Nesta seção, vamos testar as capacidades de memória do agente através de uma série de interações. Vamos observar como o agente constrói contexto ao longo do tempo e relembra interações anteriores.

Primeiro, vamos nos apresentar ao agente e fazer uma pergunta de matemática:

In [ ]:
# First interaction - introduce yourself
response1 = agent("Hi, I'm John and I just enrolled in Discrete Math course. Help me solve this: How many ways can I arrange 5 books on a shelf?")
print(f"Agent: {response1}")

Vamos dar ao agente outra tarefa de cálculo:

In [ ]:
# Second interaction - another calculation
response2 = agent("I learn better with step-by-step explanation with example questions. Can you explain modular arithmetic? What's 17 mod 5?")
print(f"Agent: {response2}")

Agora, vamos ver se o agente lembra quem somos.

**Nota:** Aguarde ~20 segundos aqui para dar tempo para a memória ser extraída, consolidada e armazenada.

In [ ]:
# Third interaction - test memory recall
response3 = agent("I got that right! What's the immediate next step that I should study after modular arithmetic?")
print(f"Agent: {response3}")

Por fim, vamos verificar se o agente lembra nosso histórico de cálculos:

In [ ]:
# Fourth interaction - test context awareness
response4 = agent("This is too hard, can we try something easier?")
print(f"Agent: {response4}")

### Verificar Armazenamento de Memória

Como passo final, vamos verificar se nossas conversas foram armazenadas corretamente no AgentCore Memory. Isso demonstra que os hooks de memória estão funcionando corretamente e o agente pode acessar essas informações em interações futuras.

In [ ]:
# Check stored memories using MemorySession
try:
    namespace_prefix = f"/students/math/{ACTOR_ID}/"
    
    memories = student_session.search_long_term_memories(
        query="mathematics calculations learning progress",
        namespace_prefix=namespace_prefix,
        top_k=5
    )
    
    print(f"\n📚 Found {len(memories)} memories for student {ACTOR_ID}:")
    print("=" * 60)
    for i, memory in enumerate(memories, 1):
        if isinstance(memory, dict):
            content = memory.get('content', {})
            score = memory.get('score', 0)
            if isinstance(content, dict):
                text = content.get('text', '')[:200] + "..."
                print(f"\n{i}. [Relevance: {score:.2f}]")
                print(f"   {text}")
    print("\n" + "=" * 60)
                
except Exception as e:
    logger.error(f"Error retrieving memories: {e}")

## Funcionalidades Avançadas: Ramificação e Metadata

### Ramificação de Conversas

A ramificação permite explorar caminhos alternativos de conversa a partir de qualquer ponto. Isso é útil para:
- Testar diferentes níveis de dificuldade
- Explorar explicações alternativas
- Teste A/B de abordagens de ensino

Vamos criar uma ramificação para explorar um caminho de problemas mais difíceis:

In [ ]:
# Get the last event ID from our conversation
events = student_session.list_events()
if events:
    last_event_id = events[-1].eventId
    
    # Fork conversation to explore advanced topics
    branch_event = student_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="advanced-path",
        messages=[
            ConversationalMessage(
                "Actually, I'm ready for a challenge! Can you give me a harder problem involving modular arithmetic and combinatorics?",
                USER
            ),
            ConversationalMessage(
                "Great! Here's a challenging problem: How many 4-digit numbers are there where the sum of digits is congruent to 3 (mod 5)? This combines modular arithmetic with counting principles.",
                ASSISTANT
            )
        ]
    )
    
    logger.info(f"✅ Created branch 'advanced-path' from event {last_event_id}")
    logger.info(f"   Branch event ID: {branch_event['eventId']}")
    
    # List all branches
    branches = student_session.list_branches()
    print(f"\n🌳 Session has {len(branches)} branch(es):")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count} events")
    
    # Get events from the advanced branch
    advanced_events = student_session.list_events(branch_name="advanced-path")
    print(f"\n📋 Advanced branch has {len(advanced_events)} events")
else:
    print("No events found to branch from")

### Uso Criativo de Metadata

Metadata permite marcar eventos com informações personalizadas para melhor organização e recuperação. Vamos usar metadata para rastrear:
- Níveis de dificuldade dos problemas
- Desempenho do estudante
- Categorias de tópicos
- Marcos de aprendizado

In [ ]:
from bedrock_agentcore.memory.models import StringValue

# Add a new interaction with rich metadata
metadata_event = student_session.add_turns(
    messages=[
        ConversationalMessage(
            "Let me try: If I choose 3 books from 5, that's C(5,3) = 10 ways, right?",
            USER
        ),
        ConversationalMessage(
            "Excellent! You correctly applied the combination formula. That's exactly right: C(5,3) = 5!/(3!×2!) = 10.",
            ASSISTANT
        )
    ],
    metadata={
        "difficulty": StringValue.build("intermediate"),
        "topic": StringValue.build("combinatorics"),
        "subtopic": StringValue.build("combinations"),
        "performance": StringValue.build("correct"),
        "milestone": StringValue.build("first_correct_combination"),
        "learning_stage": StringValue.build("applying_formulas")
    }
)

logger.info(f"✅ Added event with metadata - Event ID: {metadata_event['eventId']}")
print("\n📊 Event tagged with:")
print("   - Difficulty: intermediate")
print("   - Topic: combinatorics")
print("   - Performance: correct")
print("   - Milestone: first_correct_combination")

### Consultando Eventos por Metadata

Agora podemos filtrar eventos com base em metadata para analisar o progresso do estudante:

In [ ]:
from bedrock_agentcore.memory.models import EventMetadataFilter, LeftExpression, RightExpression, OperatorType

# Query events where student got the answer correct
try:
    correct_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "performance"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "correct"}}
            }
        ]
    )
    
    print(f"\n✅ Found {len(correct_events)} event(s) where student answered correctly")
    
    # Query intermediate difficulty problems
    intermediate_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "difficulty"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "intermediate"}}
            }
        ]
    )
    
    print(f"📈 Found {len(intermediate_events)} intermediate difficulty problem(s)")
    
    # Query combinatorics topics
    combinatorics_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "topic"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "combinatorics"}}
            }
        ]
    )
    
    print(f"🎯 Found {len(combinatorics_events)} combinatorics-related event(s)")
    
    print("\n💡 Use cases for metadata:")
    print("   - Track student progress by difficulty level")
    print("   - Identify topics needing more practice")
    print("   - Generate performance reports")
    print("   - Personalize learning paths based on history")
    
except Exception as e:
    logger.error(f"Error querying metadata: {e}")
    print(f"Note: Metadata filtering requires events with metadata tags")

Tutorial concluído! 🎉

Principais aprendizados:
- Hooks de memória armazenam e recuperam automaticamente o contexto da conversa
- Agentes podem manter estado entre múltiplas interações
- AgentCore Memory fornece busca semântica para contexto relevante
- Ferramentas podem ser combinadas com memória para funcionalidades aprimoradas
- **Ramificação permite explorar caminhos alternativos de conversa**
- **Metadata fornece capacidades poderosas de filtragem e análise**

## Limpeza

### Opcional: Excluir Recurso de Memória

Após concluir o tutorial, você pode querer excluir o recurso de memória para evitar custos desnecessários. O código a seguir é fornecido para limpeza, mas está comentado por padrão.

In [ ]:
# Uncomment to delete the memory resource
# try:
#     memory_manager.delete_memory(memory_id)
#     print(f"✅ Deleted memory resource: {memory_id}")
# except Exception as e:
#     print(f"Error deleting memory: {e}")